In [234]:
import torch 
import torch.nn as nn
torch.set_printoptions(threshold=float('inf'))
torch.set_printoptions(sci_mode=False)
# init memory and buffer
mem_A = torch.zeros(31008, 16)
mem_B = torch.zeros(31008, 16)
mem_w = torch.zeros(585, 16)
reg_b = torch.zeros(5, 16)

input_buffer = torch.zeros(3, 4, 16)
weight_buffer = torch.zeros(3, 3, 16)
output_buffer = torch.zeros(3, 16)

In [112]:
model = torch.load('Model_May27_0439.pt', map_location='cpu')

In [ ]:
def padding(memory): # for memory A and B
    address, word = memory.size()
    for addr in range(address):
        # pad vertically
        if (addr%W==0) or (addr%W==W-1): # (addr%102==0) or (addr%102==101)
            memory[addr, :] = torch.zeros(word)
        # pad horizontally
        if (addr >= 0 and addr <= W-1) or ( addr >= (H-1)*W and addr <=W*H): # (addr >= 0 and addr <= 101) or ( addr >= 10302 and addr <=10403) 
            memory[addr, :] = torch.zeros(word)

In [235]:
class init_SRAM_write():
    def __call__(self, img, mask):
        for i in range(3): # for R, G, B
            for j in range(100): # H
                for k in range(100): # W
                    mem_A[i*(H-1)*W+W+1+H*j+k, 0] = img[0, i:i+1, j:j+1, k:k+1] #channel 1 = img
                    mem_A[i*(H-1)*W+W+1+H*j+k, 1] = mask[0, i:i+1, j:j+1, k:k+1] #channel 2 = pixel mask
        print('memory A write done')
        
class init_weight_write():
    def __init__(self,):
        self.conv1_weight = model['shared_block.conv1.weight']
        self.conv1_bias = model['shared_block.conv1.bias']
        self.conv2_weight=model['shared_block.conv2.weight']
        self.conv2_bias=model['shared_block.conv2.bias']
        self.conv3_weight=model['shared_block.conv3.weight']
        self.conv3_bias=model['shared_block.conv3.bias']
        self.conv4_weight=model['shared_block.conv4.weight']
        self.conv4_bias=model['shared_block.conv4.bias']
        self.conv5_weight=model['shared_block.conv5.weight']
        self.conv5_bias=model['shared_block.conv5.bias']
    def __call__(self,):
        mem_w[0:144, 0:1]=self.conv1_weight[:, 0, :, :].reshape(144, 1)
        mem_w[0:144, 1:2]=self.conv1_weight[:, 1, :, :].reshape(144, 1)
        mem_w[144:288]=self.conv2_weight.permute(0,2,3,1).reshape(144, 16)
        mem_w[288:432]=self.conv3_weight.permute(0,2,3,1).reshape(144, 16)
        mem_w[432:576]=self.conv4_weight.permute(0,2,3,1).reshape(144, 16)
        mem_w[576:585]=self.conv5_weight.permute(0,2,3,1).reshape(9, 16)
        reg_b[0, :]=self.conv1_bias
        reg_b[1, :]=self.conv2_bias
        reg_b[2, :]=self.conv3_bias
        reg_b[3, :]=self.conv4_bias
        reg_b[4, 0:2]=self.conv5_bias
        print('weight write done')
        
        
class layer1 ():
    def __call__(self,):
        print("layer 1 done")
        
class layer2 ():
    def __call__(self,):
        print("layer 2 done")
        
class layer3 ():
    def __call__(self,):
        print("layer 3 done")
        
class layer4 ():
    def __call__(self,):
        print("layer 4 done")
        
class layer5 ():
    def __call__(self,):
        print("layer 5 done")
        
class layer6 ():
    def __call__(self,):
        print("layer 6 done")


In [236]:
# initial state
state = 'idle'
initial_SRAMw_done=False
initial_weight_done=False
layer_done=False

# init layer classes
layer1_en = layer1()
layer2_en = layer2()
layer3_en = layer3()
layer4_en = layer4()
layer5_en = layer5()
layer6_en = layer6()
SRAM_write = init_SRAM_write()
weight_write = init_weight_write()

# image & pixel mask
input_img = torch.randn(1, 3, 100, 100) # R, G, B image with size 100 x 100
pixel_mask = torch.randn(1, 3, 100, 100)
img_H = 100
img_W = 100
pad_H = 1
pad_W = 1
B, C, H, W = 1, 2, img_H+pad_H*2, img_W+pad_W*2

# Main FSM 
#IDLE
print("Process Started")

#S_SRAM_W
SRAM_write(input_img, pixel_mask)
weight_write()

#Layer 1~6
layer1_en()
layer2_en()
layer3_en()
layer4_en()
layer5_en()
layer6_en()

#IDLE
print("Process Finished")

Process Started
memory A write done
weight write done
layer 1 done
layer 2 done
layer 3 done
layer 4 done
layer 5 done
layer 6 done
Process Finished
